# Lab 3: Finding and Loading Your Data

The most common reason deep learning projects fail is not a bad model — 
it is bad data. Too little of it, wrong labels, unexpected class imbalance, 
images that look nothing like what the model will see at test time.

Before you write a single line of model code, you need to understand your dataset.

This lab covers:
1. Where to find datasets for your project
2. Loading images into PyTorch — the two main approaches
3. Essential sanity checks every project needs
4. Spotting common data problems early

After the notebook, you will apply these tools to your own dataset 
and fill in the data assessment worksheet.

**Bring your week 2 scoping worksheet — you will need it today.**

## Setup

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from collections import Counter
import urllib.request
import zipfile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Where to find datasets

Before collecting your own data, always check whether a suitable dataset 
already exists. Collecting and labelling images is time-consuming, and 
existing datasets often come with established baselines you can compare against.

**Good places to look:**

| Source | Best for | Link |
|---|---|---|
| **torchvision.datasets** | Standard benchmarks (ImageNet, CIFAR, MNIST...) | pytorch.org/vision/stable/datasets.html |
| **HuggingFace Datasets** | Huge variety, easy to load | huggingface.co/datasets |
| **Kaggle** | Competition datasets, many domains | kaggle.com/datasets |
| **Papers with Code** | Datasets linked to published papers | paperswithcode.com/datasets |
| **Roboflow Universe** | Computer vision datasets, annotated | universe.roboflow.com |
| **Google Dataset Search** | General purpose search engine | datasetsearch.research.google.com |
| **Zenodo / Figshare** | Scientific and medical datasets | zenodo.org |

**Before using any dataset, check:**
- Licence: can you use it for a course project?
- Size: is there enough data for your task?
- Labels: are they the right granularity for your problem?
- Quality: are the images consistent with what your model will see at test time?

Let's download a sample dataset to work with today:

In [ ]:
# Download the cats/dogs/horses dataset from week 2
import gdown
gdown.download('https://drive.google.com/uc?id=1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj', 'data.zip', quiet=False)
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')

data_dir = Path('data')
print('Dataset ready.')

## 2. Loading images — two approaches

### Approach A: ImageFolder

The simplest approach. If your images are already organised into 
one folder per class, `datasets.ImageFolder` handles everything automatically.

```
data/
  train/
    cat/   img001.jpg  img002.jpg  ...
    dog/   img001.jpg  img002.jpg  ...
  val/
    cat/   img001.jpg  ...
    dog/   img001.jpg  ...
```

This is the format used for most standard datasets. **If you can get 
your data into this format, use ImageFolder** — it saves a lot of code.

In [ ]:
# Standard transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load with ImageFolder — one line
train_dataset = datasets.ImageFolder(data_dir / 'train', transform=transform)

print(f'Classes:          {train_dataset.classes}')
print(f'Class to index:   {train_dataset.class_to_idx}')
print(f'Number of images: {len(train_dataset)}')

# Each item is a (tensor, label) pair
image, label = train_dataset[0]
print(f'\nFirst item:')
print(f'  Image shape: {image.shape}')   # (C, H, W)
print(f'  Label:       {label} ({train_dataset.classes[label]})')

### Approach B: Custom Dataset class

When your data does not fit the folder-per-class structure — for example, 
if labels come from a CSV file, filenames contain the label, or you need 
custom loading logic — you write your own `Dataset` class.

The pattern is always the same: inherit from `Dataset`, implement 
`__len__` and `__getitem__`.

In [ ]:
class ImageDatasetFromCSV(Dataset):
    """
    Example: dataset where labels come from a CSV file.
    CSV format:  filepath,label
                 data/train/cat/img001.jpg,0
                 data/train/dog/img002.jpg,1
    """
    def __init__(self, csv_path, transform=None):
        import csv
        self.samples   = []
        self.transform = transform
        with open(csv_path) as f:
            reader = csv.reader(f)
            next(reader)   # skip header
            for row in reader:
                self.samples.append((row[0], int(row[1])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


class ImageDatasetFromList(Dataset):
    """
    Example: dataset from lists of file paths and labels.
    The most flexible approach — adapts to almost any data structure.
    """
    def __init__(self, image_paths, labels, transform=None):
        assert len(image_paths) == len(labels)
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]


# Example: build a dataset from lists
from glob import glob

cat_images = sorted(glob(str(data_dir / 'train/cat/*.jpg')))
dog_images = sorted(glob(str(data_dir / 'train/dog/*.jpg')))

all_paths  = cat_images + dog_images
all_labels = [0] * len(cat_images) + [1] * len(dog_images)

list_dataset = ImageDatasetFromList(all_paths, all_labels, transform=transform)
print(f'Dataset from lists: {len(list_dataset)} images')
print(f'First item label: {list_dataset[0][1]} (0=cat, 1=dog)')

## 3. The DataLoader

A `DataLoader` wraps a `Dataset` and handles batching, shuffling, 
and parallel loading. You use it the same way regardless of which 
Dataset approach you chose above.

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Check one batch
images, labels = next(iter(train_loader))
print(f'Batch image shape: {images.shape}')   # (32, 3, 224, 224)
print(f'Batch label shape: {labels.shape}')   # (32,)
print(f'Labels in batch:   {labels.tolist()}')

## 4. Essential sanity checks

Run these checks on every new dataset before you start training. 
They take 5 minutes and can save you hours of debugging later.

### Check 1: Visualise samples

In [ ]:
def denormalise(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean = torch.tensor(mean).view(3,1,1)
    std  = torch.tensor(std).view(3,1,1)
    return torch.clamp(tensor * std + mean, 0, 1)

def visualise_samples(dataset, class_names, n_per_class=4):
    """
    Show n_per_class examples from each class.
    The most important sanity check — do the images and labels look right?
    """
    n_classes = len(class_names)
    fig, axes = plt.subplots(
        n_classes, n_per_class,
        figsize=(n_per_class * 3, n_classes * 3)
    )

    # Collect indices per class
    class_indices = {c: [] for c in range(n_classes)}
    for idx, (_, label) in enumerate(dataset.samples
            if hasattr(dataset, 'samples') else
            [(None, dataset[i][1]) for i in range(len(dataset))]):
        if len(class_indices[label]) < n_per_class:
            class_indices[label].append(idx)
        if all(len(v) >= n_per_class for v in class_indices.values()):
            break

    for row, (cls, indices) in enumerate(class_indices.items()):
        for col, idx in enumerate(indices[:n_per_class]):
            ax  = axes[row][col] if n_classes > 1 else axes[col]
            img = denormalise(dataset[idx][0]).permute(1,2,0).numpy()
            ax.imshow(img)
            if col == 0:
                ax.set_ylabel(class_names[cls], fontsize=11, fontweight='bold')
            ax.axis('off')

    plt.suptitle('Sample images per class — check that labels are correct', fontsize=12)
    plt.tight_layout()
    plt.show()

visualise_samples(train_dataset, train_dataset.classes)

### Check 2: Class distribution

In [ ]:
def check_class_distribution(dataset, class_names, split_name='dataset'):
    """
    Plot the number of images per class.
    Large imbalances will bias the model towards the majority class.
    """
    if hasattr(dataset, 'targets'):
        labels = dataset.targets
    else:
        labels = [dataset[i][1] for i in range(len(dataset))]

    counts = Counter(labels)
    names  = [class_names[i] for i in sorted(counts.keys())]
    values = [counts[i] for i in sorted(counts.keys())]
    total  = sum(values)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Bar chart
    bars = ax1.bar(names, values, color='steelblue', edgecolor='white')
    ax1.set_title(f'Class distribution — {split_name}')
    ax1.set_ylabel('Number of images')
    for bar, val in zip(bars, values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', va='bottom', fontsize=10)

    # Pie chart showing balance
    ax2.pie(values, labels=names, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Class balance')

    plt.tight_layout()
    plt.show()

    print(f'Total: {total} images')
    print(f'Min class: {min(values)} ({names[values.index(min(values))]})')
    print(f'Max class: {max(values)} ({names[values.index(max(values))]})')
    ratio = max(values) / max(min(values), 1)
    if ratio > 3:
        print(f'WARNING: Class imbalance ratio {ratio:.1f}x — consider resampling or weighted loss.')
    else:
        print(f'Class imbalance ratio: {ratio:.1f}x — looks balanced.')

check_class_distribution(train_dataset, train_dataset.classes, 'training set')

### Check 3: Image size distribution

In [ ]:
def check_image_sizes(image_dir, sample_size=200):
    """
    Check the distribution of original image sizes before resizing.
    Extreme size variation can indicate mixed data sources or quality issues.
    """
    widths, heights = [], []
    image_files = list(Path(image_dir).rglob('*.jpg')) + \
                  list(Path(image_dir).rglob('*.png')) + \
                  list(Path(image_dir).rglob('*.jpeg'))

    # Sample a subset for speed
    import random
    sampled = random.sample(image_files, min(sample_size, len(image_files)))

    for path in sampled:
        try:
            with Image.open(path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
        except Exception:
            pass   # skip unreadable files

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(widths,  bins=20, color='steelblue', edgecolor='white')
    axes[0].set_title('Image widths'); axes[0].set_xlabel('Pixels')

    axes[1].hist(heights, bins=20, color='coral',     edgecolor='white')
    axes[1].set_title('Image heights'); axes[1].set_xlabel('Pixels')

    plt.suptitle(f'Image size distribution (sample of {len(widths)})', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f'Width:  min={min(widths)}, max={max(widths)}, median={int(np.median(widths))}')
    print(f'Height: min={min(heights)}, max={max(heights)}, median={int(np.median(heights))}')

    if max(widths) / max(min(widths), 1) > 10:
        print('WARNING: Very large variation in image sizes — check for outliers.')

check_image_sizes(data_dir / 'train')

### Check 4: Tensor value range after normalisation

In [ ]:
def check_tensor_stats(loader, n_batches=5):
    """
    Check the mean and std of pixel values after normalisation.
    After ImageNet normalisation, mean should be near 0 and std near 1.
    Large deviations suggest the wrong normalisation was applied.
    """
    all_means = []
    all_stds  = []

    for i, (images, _) in enumerate(loader):
        if i >= n_batches:
            break
        all_means.append(images.mean(dim=[0,2,3]).numpy())  # per channel
        all_stds.append(images.std(dim=[0,2,3]).numpy())

    mean = np.array(all_means).mean(axis=0)
    std  = np.array(all_stds).mean(axis=0)

    print('After normalisation (should be near mean≈0, std≈1 per channel):')
    for c, name in enumerate(['R', 'G', 'B']):
        print(f'  {name}: mean={mean[c]:.3f}, std={std[c]:.3f}')

    if abs(mean).max() > 0.5:
        print('WARNING: Mean far from 0 — check normalisation values.')
    if abs(std - 1).max() > 0.5:
        print('WARNING: Std far from 1 — check normalisation values.')

check_tensor_stats(train_loader)

### Check 5: Train / validation split

In [ ]:
def check_split(train_dataset, val_dataset, class_names):
    """
    Verify the train/val split looks reasonable.
    The split should be consistent across classes.
    """
    train_counts = Counter(
        train_dataset.targets if hasattr(train_dataset, 'targets')
        else [train_dataset[i][1] for i in range(len(train_dataset))]
    )
    val_counts = Counter(
        val_dataset.targets if hasattr(val_dataset, 'targets')
        else [val_dataset[i][1] for i in range(len(val_dataset))]
    )

    print(f'{'Class':<20} {'Train':>8} {'Val':>8} {'Val %':>8}')
    print('-' * 46)
    for i, name in enumerate(class_names):
        tr  = train_counts[i]
        vl  = val_counts[i]
        pct = vl / (tr + vl) * 100 if (tr + vl) > 0 else 0
        print(f'{name:<20} {tr:>8} {vl:>8} {pct:>7.1f}%')

    total_train = sum(train_counts.values())
    total_val   = sum(val_counts.values())
    total_pct   = total_val / (total_train + total_val) * 100
    print('-' * 46)
    print(f'{'TOTAL':<20} {total_train:>8} {total_val:>8} {total_pct:>7.1f}%')

    if total_pct < 10:
        print('WARNING: Validation set is very small — results may be noisy.')
    if total_pct > 40:
        print('WARNING: Validation set is large — consider using more data for training.')

val_dataset = datasets.ImageFolder(data_dir / 'val', transform=transform)
check_split(train_dataset, val_dataset, train_dataset.classes)

## 5. Common data problems to watch for

Here are the data issues that most commonly derail student projects, 
with how to spot and fix them:

| Problem | How to spot it | What to do |
|---|---|---|
| **Label noise** | Low accuracy despite large dataset; model confidently wrong | Review sample images per class; re-label or filter |
| **Class imbalance** | Imbalance ratio > 5x | Weighted loss, oversampling, or undersampling |
| **Train/val leakage** | Val accuracy much higher than expected | Check that val images are truly unseen |
| **Distribution shift** | Good val accuracy, poor real-world performance | Ensure val set matches deployment conditions |
| **Wrong normalisation** | Loss does not decrease, or decreases very slowly | Check tensor stats (Check 4 above) |
| **Corrupt images** | Errors during loading or training | Filter with `Image.verify()` |
| **Too few images** | Overfitting with high train acc, low val acc | Collect more data or use stronger augmentation |
| **Wrong colour format** | Images look right but model performs poorly | Ensure RGB conversion (`.convert('RGB')`) |

In [ ]:
def find_corrupt_images(image_dir):
    """
    Scan for images that cannot be opened.
    Corrupt images will crash training with cryptic errors.
    Run this once before you start training.
    """
    image_files = list(Path(image_dir).rglob('*.jpg')) + \
                  list(Path(image_dir).rglob('*.png')) + \
                  list(Path(image_dir).rglob('*.jpeg'))
    corrupt = []
    for path in image_files:
        try:
            with Image.open(path) as img:
                img.verify()
        except Exception as e:
            corrupt.append((path, str(e)))

    if corrupt:
        print(f'Found {len(corrupt)} corrupt image(s):')
        for path, err in corrupt:
            print(f'  {path}: {err}')
    else:
        print(f'Scanned {len(image_files)} images — no corrupt files found.')
    return corrupt

corrupt = find_corrupt_images(data_dir)

## 6. Putting it all together — your project dataset

Now apply what you have learned to your own project dataset.

Work through these steps with your group:

**Step 1 — Load your dataset.**
Use `ImageFolder` if possible, or adapt `ImageDatasetFromList` if not.
If you do not have a dataset yet, identify one today — see the sources 
in section 1 and pick the most promising candidate.

**Step 2 — Run all five sanity checks.**
Copy the check functions above and run them on your data.
Write down what you find.

**Step 3 — Fill in the data assessment worksheet.**
Use what you have found in steps 1 and 2.
A TA will review it with you before the end of the session.

**Questions to answer before you leave:**
- Do you have enough data? (Rough guide: at least 100 images per class for transfer learning)
- Is there a class imbalance problem?
- Is the data quality good enough?
- Does your validation set reflect how the model will be used?